# Notebook 06 — Genotoxicity & Mutagenicity (ICH M7 Framework)
**Author: Himanshu Goel** | [Website](https://hgoelgithub.github.io)

Genotoxicity testing is **mandatory** for all new drugs (ICH M3(R2)). The standard battery:
1. **Ames test** (bacterial mutagenicity, OECD 471) — most important
2. **Micronucleus test** (chromosomal damage, OECD 487)
3. **In vitro chromosomal aberration** (OECD 473)

**ICH M7(R2)** accepts two complementary QSAR models as an alternative to Ames testing for pharmaceutical impurity assessment — one of the highest regulatory recognitions of in silico toxicology.

Key datasets: Hansen mutagenicity (6512 compounds), ISSSTY (5000+), Tox21 SR-ATAD5

In [ ]:
!pip install rdkit scikit-learn xgboost pandas numpy matplotlib -q

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, rdMolDescriptors, MACCSkeys
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
import warnings; warnings.filterwarnings('ignore')

# Ames mutagenicity dataset (representative subset)
ames = [
    ("c1ccc2c(c1)ccc1cccc3cccc2c13",1,"Benzo[a]pyrene","PAH"),
    ("[O-][N+](=O)c1ccccc1",        1,"Nitrobenzene","Nitroaromatic"),
    ("ClCCCl",                       1,"1,2-DCE","Alkylating"),
    ("BrCCBr",                       1,"EDB","Alkylating"),
    ("Nc1ccc([N+](=O)[O-])cc1",     1,"4-Nitroaniline","Nitroaro+amine"),
    ("c1ccc2ncccc2c1",               1,"Quinoline","Aromatic N-heterocycle"),
    ("CCCCBr",                       1,"1-Bromobutane","Alkylating"),
    ("O=Cc1ccccc1",                  1,"Benzaldehyde","Electrophile"),
    ("c1ccc(N)cc1",                  1,"Aniline","Aromatic amine"),
    ("NN",                           1,"Hydrazine","DNA alkylating"),
    ("C=O",                          1,"Formaldehyde","Direct alkylating"),
    ("c1ccc2cc3ccccc3cc2c1",         1,"Pyrene","PAH"),
    ("OC(=O)c1ccccc1",              0,"Benzoic acid","Safe"),
    ("CC(=O)Oc1ccccc1C(=O)O",      0,"Aspirin","Safe"),
    ("CN(C)C(=N)NC(=N)N",          0,"Metformin","Safe"),
    ("Nc1ccccc1NC(=O)c1ccccc1",    0,"Benzanilide","Low concern"),
    ("OCC(O)CO",                    0,"Glycerol","Safe"),
    ("CC(=O)OCC",                   0,"Ethyl acetate","Safe"),
    ("CC(C)=O",                     0,"Acetone","Safe"),
    ("CC#N",                        0,"Acetonitrile","Safe"),
    ("C1CCCCC1",                    0,"Cyclohexane","Safe"),
    ("OC(=O)CC(=O)O",              0,"Malonic acid","Safe"),
    ("Clc1ccccc1Cl",                0,"1,2-DCB","Low"),
    ("CC(C)(C)OC(=O)O",            0,"Boc-OH","Safe"),
]

def geno_features(smi):
    mol=Chem.MolFromSmiles(smi)
    if not mol: return None
    ecfp=np.array(AllChem.GetMorganFingerprintAsBitVect(mol,2,2048))
    maccs=np.array(MACCSkeys.GenMACCSKeys(mol))
    pc=np.array([
        Descriptors.ExactMolWt(mol), Descriptors.MolLogP(mol),
        rdMolDescriptors.CalcNumAromaticRings(mol),
        sum(1 for a in mol.GetAtoms() if a.GetAtomicNum()==7),
        sum(1 for a in mol.GetAtoms() if a.GetAtomicNum()==17),
        sum(1 for a in mol.GetAtoms() if a.GetAtomicNum()==35),
        Descriptors.NHOHCount(mol), Descriptors.NOCount(mol),
        rdMolDescriptors.CalcNumRings(mol),
    ])
    return np.concatenate([ecfp,maccs,pc])

valid=[(s,l,n,c) for s,l,n,c in ames if geno_features(s) is not None]
X=np.array([geno_features(s) for s,_,_,_ in valid])
y=np.array([l for _,l,_,_ in valid])
names=[n for _,_,n,_ in valid]
scaler=StandardScaler(); X_s=scaler.fit_transform(X)
print(f"Ames dataset: {len(y)} | Mutagens: {y.sum()} | Non-mutagens: {(y==0).sum()}")

## ICH M7(R2) Structural Alert Classes

In [ ]:
ich_m7={
    "Class1|Epoxide":        "[OX2r3]",
    "Class1|Alkyl halide":   "[CX4;H2][Cl,Br,I]",
    "Class1|Alkyl sulfonate":"[CX4]OS(=O)(=O)",
    "Class1|Acyl halide":    "[CX3](=O)[F,Cl,Br,I]",
    "Class1|Michael acc":    "[CX3](=O)[CX3]=[CX3]",
    "Class2|Aromatic nitro": "[$([NX3](=O)=O)]c",
    "Class2|Aromatic amine": "Nc1ccccc1",
    "Class2|Hydrazine":      "[NX3][NX3]",
    "Class2|N-nitroso":      "[NX3][NX2]=O",
    "Class2|PAH":            "c1ccc2cccc3cccc1c23",
    "Class3|Aldehyde":       "[CH]=O",
    "Class3|Quinone":        "O=C1C=CC(=O)C=C1",
}
print("ICH M7(R2) Alert Screening:")
print(f"{'Compound':20s} {'Class|Alert':40s} {'Mutagenic'}")
print("-"*70)
for s,l,n,_ in valid[:15]:
    mol=Chem.MolFromSmiles(s)
    hits=[]
    for an,sm in ich_m7.items():
        try:
            p=Chem.MolFromSmarts(sm)
            if p and mol.HasSubstructMatch(p): hits.append(an)
        except: pass
    print(f"{n:20s} {hits[0] if hits else 'No alert':40s} {'YES' if l else 'No'}")

## Multi-endpoint consensus scoring (Ames + MNT + SR-ATAD5)

In [ ]:
np.random.seed(42)
# Simulate correlated MNT and SR-ATAD5 endpoints
y_mnt  =np.where(y==1,np.random.binomial(1,0.75,len(y)),np.random.binomial(1,0.1,len(y)))
y_atad5=np.where(y==1,np.random.binomial(1,0.65,len(y)),np.random.binomial(1,0.15,len(y)))

cv=StratifiedKFold(4,shuffle=True,random_state=42)
print("Multi-endpoint genotoxicity prediction (RF):")
for ep_name,ep_y in [("Ames",y),("MNT",y_mnt),("SR-ATAD5",y_atad5)]:
    if ep_y.std()>0:
        s=cross_val_score(RandomForestClassifier(300,class_weight='balanced',random_state=42),
                          X_s,ep_y,cv=cv,scoring='roc_auc')
        print(f"  {ep_name:10s}  AUC={s.mean():.3f}  Pos={ep_y.sum()}")

# Consensus prediction
rf_a=RandomForestClassifier(300,class_weight='balanced',random_state=42).fit(X_s,y)
rf_m=RandomForestClassifier(300,class_weight='balanced',random_state=42).fit(X_s,y_mnt)
rf_t=RandomForestClassifier(300,class_weight='balanced',random_state=42).fit(X_s,y_atad5)
pa=rf_a.predict_proba(X_s)[:,1]
pm=rf_m.predict_proba(X_s)[:,1]
pt=rf_t.predict_proba(X_s)[:,1]
consensus=(pa+pm+pt)/3

fig,ax=plt.subplots(figsize=(9,5))
from matplotlib.cm import get_cmap
sc=ax.scatter(pa,pm,c=consensus,cmap='RdYlGn_r',s=80,vmin=0,vmax=1,zorder=5)
for i,(n,a,m) in enumerate(zip(names,pa,pm)):
    ax.annotate(n,(a,m),fontsize=6.5,xytext=(3,3),textcoords='offset points')
plt.colorbar(sc,ax=ax,label='Consensus genotox score')
ax.axvline(0.5,color='gray',linestyle='--',lw=0.8)
ax.axhline(0.5,color='gray',linestyle='--',lw=0.8)
ax.set_xlabel("P(Ames+)"); ax.set_ylabel("P(MNT+)")
ax.set_title("Consensus Genotoxicity — Ames vs MNT")
plt.tight_layout(); plt.savefig("genotox_consensus.png",dpi=150); plt.show()

## Key takeaways
- ICH M7(R2) accepts two complementary QSAR models for pharmaceutical impurity assessment
- Consensus scoring across Ames+MNT+SR-ATAD5 reduces false negatives (critical for safety)
- Class 1 alerts (epoxides, alkyl halides) are highest priority — direct DNA reactivity
- PAHs are consistently the hardest class — require metabolic activation (CYP1A1/1B1)
- Industry tools: Derek Nexus (Lhasa), Sarah Nexus, TOPKAT, VEGA — all ICH M7 compliant
- AUC target: > 0.85 for regulatory-grade mutagenicity models (OECD QSAR guidance)